In [ ]:
# !pip install beautifulsoup4
# !pip install langchain_google_genai
# !pip install rdflib

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from bs4 import BeautifulSoup
import re

import agent_generate_classes
import agent_generate_predicates
import agent_alignment_classes
import agent_generate_hierarchy

import json
import load_onto


/home/thibaut/miniconda3/envs/Sahar/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
root = "./"

PATH_OUTPUT = root+"Outputs/"
PATH_ONTOLOGIES = root+"Ontologies/"

with open(root+"OJ_L_202401689_EN_TXT.html") as fp:
    soup = BeautifulSoup(fp)


In [ ]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key="..."
)

# Retrieve pre-made definitions

In [3]:
description_box = soup.find('div', attrs={'id': 'art_3'})
texts = description_box.find_all("p", attrs={"class":"oj-normal"})

class Definition():
  def __init__(self, id, text, parent=None):
    self.id = id
    self.text = text
    self.parent = parent
    self.children = list()
    self.title = None
    self.description = None

    self.format_definition()

  def add_child(self, child):
    self.children.append(child)

  def get_parent(self):
    return self.parent

  def __str__(self):
    return self.id + ": " + self.text

  def print_definition_and_children(self):
    res = ""
    if self.title:
      res += self.title + "---"
    if self.description:
      res += self.description
    for child in self.children:
      res += "\n" + child.print_definition_and_children()
    return res
  
  def return_definitions_rec(self):
    description = self.description
    for child in self.children:
      description += "\n"+child.return_definitions_rec()
    return description
  
  def return_title(self):
    return self.title

  def format_definition(self):
    matched = re.search(r"[‘][A-Za-z-_\s]+[’]", self.text)
    if matched:
      self.title = matched.group(0)[1:-1].replace("\n","")
      self.description = self.text.replace(matched.group(0), "").replace("\n", "").replace("\xa0","")
    else:
      self.description = self.text.replace("\n", "").replace("\xa0","")

last_was_id = False
last_id = None
previous_depth = 0
current_depth = 0
root_definitions = Definition("Root", "This is the root of the definitions")
current_definitions = root_definitions

for text_raw in texts:
  if last_was_id == True:
    last_was_id = False

    if previous_depth >= current_depth:
      for i in range(previous_depth - current_depth+1):
        current_definitions = current_definitions.get_parent()
        previous_depth -= 1

    child = Definition(last_id, text_raw.get_text(), current_definitions)
    current_definitions.add_child(child)
    current_definitions = child
    previous_depth = current_depth

  else:
    if (text := text_raw.get_text())[0] == "(":
      last_was_id = True
      if re.fullmatch(r"\([0-9]+\)", text):
        current_depth = 1
        last_id = text
      elif re.fullmatch(r"\([a-z]+\)", text):
        current_depth = 2
        last_id = text
      elif re.fullmatch(r"\(i+\)",text):
        current_depth = 3
        last_id = text
      else:
        print("Match not taken into account")
        print(i)

# Generate new definitions based on the original ones

In [ ]:
extracted_articles = ""
for i in range(1,114):
    extracted_articles+=soup.find('div', attrs={"id": f"art_{i}"}).__str__()

prompt = f"""### Role
Expert Legal Ontologist

### Context
I have already extracted a set of core classes from the document using a standard extraction pipeline. 
These are the **Defined Classes** (DO NOT RE-EXTRACT OR REDEFINE THESE):
{"\n".join([class_hardcoded.print_definition_and_children() for class_hardcoded in root_definitions.children])}

### Task
Analyze the provided legal document to find **New, Latent, or Implied Classes** that were not caught by the initial extraction but are critical for litigation. 

### Guidelines for New Classes
1. **Uniqueness**: Ensure the new class does not overlap in meaning with the Defined Classes listed above.
2. **Litigation Value**: Focus on "Functional Roles" (e.g., Indemnitor) or "Status Conditions" (e.g., Default State) that a lawyer would use to argue a breach.
3. **Behavioral Definition**: The definition must explain the specific legal trigger or consequence of this class within this document.

### Input Document
"""
# soup.find('div', attrs={'id': 'art_5'}).__str__()
#+soup.__str__()#+soup.find('div', attrs={'id': 'art_5'}).__str__()

In [ ]:
model_generate_classes = agent_generate_classes.set_structured_output(model=model)
res = agent_generate_classes.apply_prompt(model_generate_classes, prompt, extracted_articles)

In [ ]:
class ClassOntology():
    def __init__(self, label:str, comment:str, is_gen:bool):
        self.label = label
        self.comment = comment
        self.is_gen = bool

In [ ]:
set_new_class = {(new_class.name, new_class.definition) for new_class in res.classes_ontology}

In [ ]:
len(set_new_class)

20

In [ ]:
classes_to_ouput = dict()
id = 0 
for child in root_definitions.children:
    classes_to_ouput[id] = {
        "label": child.return_title(),
        "description": child.return_definitions_rec(),
        "source": "Doc"
    }
    id+=1

for class_gen in set_new_class:
    classes_to_ouput[id] = {
        "label": class_gen[0],
        "description":class_gen[1],
        "source":"Gen"
    }
    id+=1
    
with open(PATH_OUTPUT+"classes.json", "w") as json_file:
    json.dump(classes_to_ouput, json_file, indent=4)


In [ ]:
with open(PATH_OUTPUT+"classes.json", "r") as json_file:
    classes_to_ouput = json.load(json_file)

In [ ]:
class_gen_labels = {class_gen["label"] for class_gen in classes_to_ouput.values()}

In [ ]:

classes_external = load_onto.retrieve_information_classes(PATH_ONTOLOGIES)

Ontologies/vair.ttl
	Classes found:858
Ontologies/airo.ttl
	Classes found:46


In [ ]:
prompt = f"""### Role
You are an expert Ontologist and Knowledge Graph Architect specializing in Legal Informatics. Your goal is to map a set of extracted legal concepts to established industry ontologies.

### Context 
You are performing a mapping task using two specific RDFS/OWL relations:
1. **subClassOf**: Used when an extracted class is a more specific instance of an external class.
1. **higherClassOf**: Used when an extracted class is a more general instance of an external class.
2. **equivalentClass**: Used when an extracted class and an external class represent the exact same concept.

### Classes to Map (Extracted from Document)
{"\n".join([f"Class name: {class_legal["label"]} Description: {class_legal["description"]}" for class_legal in classes_to_ouput.values()])}

### External Reference Ontologies
Below are the definitions from external standards (e.g., AIRO, DPV, or the EU AI Act official terminology):
{"\n".join([f"Class URI: {iri} Class label: {class_external["label"]} Description: {class_external["comment"]}" for iri, class_external in classes_external.items()])}


### Task 
1. Analyze each Extracted Class against the External Reference Ontologies.
2. Identify the most logically sound relation (subClassOf, higherClassOf or equivalentClass).
3. If no clear relationship exists, do not guess; omit the relation."""

In [ ]:

model_alignment_classes = agent_alignment_classes.set_structured_output(model=model)

res = agent_alignment_classes.apply_prompt(model_alignment_classes, prompt)

In [ ]:
res_anchoring = dict() 
id=0
for anchor in res.anchoring_relations:
    res_anchoring[id] = {
        "prop":anchor.name,
        "sub":anchor.subject,
        "obj":anchor.object,
        "iri":anchor.object_iri,
        "justification":anchor.justification
    }
    id+=1

In [ ]:
res_anchoring

{0: {'prop': 'equivalentClass',
  'sub': 'AI system',
  'obj': 'AI System',
  'iri': 'https://w3id.org/airo#AISystem',
  'justification': "The definition of 'AI system' from the document is identical to the definition of 'AI System' in the AIRO ontology."},
 1: {'prop': 'equivalentClass',
  'sub': 'risk',
  'obj': 'Risk',
  'iri': 'https://w3id.org/airo#Risk',
  'justification': "The extracted definition of 'risk' (combination of probability of harm and severity) aligns directly with the core concept of 'Risk' in AIRO, which is expressed in terms of likelihood and severity."},
 2: {'prop': 'equivalentClass',
  'sub': 'provider',
  'obj': 'AI Provider',
  'iri': 'https://w3id.org/airo#AIProvider',
  'justification': "The definition of 'provider' from the document is identical to the definition of 'AI Provider' in the AIRO ontology."},
 3: {'prop': 'equivalentClass',
  'sub': 'deployer',
  'obj': 'AI Deployer',
  'iri': 'https://w3id.org/airo#AIDeployer',
  'justification': "The definiti

In [ ]:
with open(f"{PATH_OUTPUT}anchoring.json", "w", encoding="UTF-8") as file_json:
    json.dump(res_anchoring, file_json, indent=4)

In [ ]:
with open(PATH_OUTPUT+"anchoring.json", "r") as json_file:
    res_anchoring = json.load(json_file)

# Predicates

In [ ]:
prompt = f"""### Role
You are an expert Ontologist and Knowledge Graph Architect specializing in Legal Informatics.

### Context
You have already identified the following Classes (Nodes) within the legal document:
{"\n".join([f"Class name: {class_legal["label"]} Description: {class_legal["description"]}" for class_legal in classes_to_ouput.values()])}

### Task
Extract all functional **Relations (Edges)** between these classes as found in the text and solely these classes. Format these as RDF-style triples: (Subject) -> [Predicate] -> (Object).

### Objectives
1. **Identify Predicates**: Find the actions, obligations, or ownership links connecting the defined classes.
2. **Define Constraints**: 
    * **Domain**: The class that acts as the source of the relation.
    * **Range**: The class that acts as the target of the relation.
3. **Minimalism**: Only provide a Domain or Range if the document explicitly necessitates that specific class name. If a relation is generic, leave the field empty.

### Output Requirements
For every relation:
* **Name**: Use camelCase for the predicate (e.g., isBoundBy, providesNoticeTo).
* **Definition**: A brief explanation of the legal trigger for this link.
* **Domain**: Must strictly match the names of the Classes name provided in the Context above with none is applicable.
* **Range**: Must strictly match the names of the Classes name provided in the Context above with none is applicable.

### Input Document
"""

In [ ]:
print(prompt)
model_generate_predicates = agent_generate_predicates.set_structured_output(model=model)
res_predicates_generation = agent_generate_predicates.apply_prompt(model_generate_predicates, prompt, extracted_articles)

### Role
You are an expert Ontologist and Knowledge Graph Architect specializing in Legal Informatics.

### Context
You have already identified the following Classes (Nodes) within the legal document:
Class name : AI system Description :  means amachine-based system that is designed to operate with varying levels of autonomy and that may exhibit adaptiveness after deployment, and that, for explicit or implicit objectives, infers, from the input it receives, how to generate outputs such as predictions, content, recommendations, or decisions that can influence physical or virtual environments;
Class name : risk Description :  means the combination of the probability of an occurrence of harm and the severity of that harm;
Class name : provider Description :  means anatural or legal person, public authority, agency or other body that develops an AI system or ageneral-purpose AI model or that has an AI system or ageneral-purpose AI model developed and places it on the market or puts the AI 

OutputParserException: Failed to parse RelationCollection from completion {"relations_ontology": [{"name": "regulatesPlacingOnMarketOf", "definition": "The regulation lays down harmonised rules for the placing on the market of AI systems.", "domain": null, "range": "AI system"}, {"name": "regulatesPuttingIntoServiceOf", "definition": "The regulation lays down harmonised rules for the putting into service of AI systems.", "domain": null, "range": "AI system"}, {"name": "regulatesUseOf", "definition": "The regulation lays down harmonised rules for the use of AI systems.", "domain": null, "range": "AI system"}, {"name": "includes", "definition": "The Regulation includes prohibitions of certain AI practices.", "domain": null, "range": "Prohibited AI Practice"}, {"name": "laysDownRequirementsFor", "definition": "The Regulation lays down specific requirements for high-risk AI systems.", "domain": null, "range": "AI system"}, {"name": "laysDownObligationsFor", "definition": "The Regulation la

In [ ]:
predicates_to_output = dict()
id = 0
for predicate in res_predicates_generation.relations_ontology:
    predicates_to_output[id]={
        "label": predicate.name,
        "description":predicate.definition,
        "domain":predicate.domain,
        "range":predicate.range,
        "source":"Generated"
    }

    id+=1

with open(PATH_OUTPUT+"predicates.json", "w") as json_file:
    json.dump(predicates_to_output, json_file, indent=4)

In [ ]:

with open(PATH_OUTPUT+"predicates.json", "r") as json_file:
    predicates_to_output = json.load(json_file)

In [ ]:
domains = {predicate["domain"] for predicate in predicates_to_output.values()}
range = {predicate["range"]  for predicate in predicates_to_output.values()}

In [ ]:
merged_predicates = dict()
for predicate in predicates_to_output.values():
    label = predicate["label"] 
    if label not in merged_predicates:
        merged_predicates[label] = {
            "label": label,
            "description":{predicate["description"]},
            "domain": {predicate["domain"]},
            "range": {predicate["range"]}
        }

    else:
        merged_predicates[label]["description"].add(predicate["description"])
        merged_predicates[label]["domain"].add(predicate["domain"])
        merged_predicates[label]["range"].add(predicate["description"])

In [ ]:
merged_predicates

{'receives': {'label': 'receives',
  'description': {'An AI system takes in input data to generate outputs.',
   'Deployers receive instructions for use for an AI system.'},
  'domain': {'AI system', 'deployer'},
  'range': {'Deployers receive instructions for use for an AI system.',
   'input data'}},
 'uses': {'label': 'uses',
  'description': {'A deployer uses an AI system under its authority.',
   'Emotion recognition systems, biometric categorisation systems, and remote biometric identification systems use biometric data for their operations.',
   'Union Market Operators (Third Country) use AI systems where the output is used in the Union.'},
  'domain': {'AI system', 'Union Market Operator (Third Country)', 'deployer'},
  'range': {'A deployer uses an AI system under its authority.',
   'Union Market Operators (Third Country) use AI systems where the output is used in the Union.',
   'biometric data'}},
 'isA': {'label': 'isA',
  'description': {'A biometric categorisation system

In [ ]:
def find_couples(elements:set) -> list:
    sorted_list = sorted([i.__str__() for i in elements])
    n = len(sorted_list)
    
    pairs = []
    
    # 2. Iterate through the list
    # The outer loop stops at n-1 because the last element has nothing after it
    for idx_i in range(n):
        for idx_j in range(idx_i + 1, n):
            i = sorted_list[idx_i]
            j = sorted_list[idx_j]
            
            # Here we apply your logic/functions
            pairs.append((i, j))

    return pairs

In [ ]:
appears_together = dict()


for predicate in merged_predicates.values():
    if len(predicate["domain"]) > 1:
        for pair in find_couples(predicate["domain"]):
            if pair not in appears_together:
                appears_together[pair] = 0
            appears_together[pair] += 1

    if len(predicate["range"]) > 1:
        for pair in find_couples(predicate["range"]):
            if pair not in appears_together:
                appears_together[pair] = 0
            appears_together[pair] += 1
            
appears_together

{('AI system', 'deployer'): 2,
 ('Deployers receive instructions for use for an AI system.', 'input data'): 1,
 ('AI system', 'Union Market Operator (Third Country)'): 1,
 ('Union Market Operator (Third Country)', 'deployer'): 2,
 ('A deployer uses an AI system under its authority.',
  'Union Market Operators (Third Country) use AI systems where the output is used in the Union.'): 1,
 ('A deployer uses an AI system under its authority.', 'biometric data'): 1,
 ('Union Market Operators (Third Country) use AI systems where the output is used in the Union.',
  'biometric data'): 1,
 ('Real-World Testing Consent (Subject)',
  'Union Market Operator (Third Country)'): 1,
 ('Real-World Testing Consent (Subject)', 'authorised representative'): 1,
 ('Real-World Testing Consent (Subject)',
  'biometric categorisation system'): 1,
 ('Real-World Testing Consent (Subject)', 'biometric data'): 1,
 ('Real-World Testing Consent (Subject)', 'deployer'): 1,
 ('Real-World Testing Consent (Subject)', 'di

# Anchoring predicates

In [ ]:
predicates_external = load_onto.retrieve_information_predicates(PATH_ONTOLOGIES)

Ontologies/vair.ttl
	Properties found:0
Ontologies/airo.ttl
	Properties found:53
Ontologies/schemaorg-current-https.ttl
	Properties found:1520


In [ ]:
prompt = f"""### Role
You are an expert Ontologist and Knowledge Graph Architect specializing in Legal Informatics. Your goal is to map a set of extracted legal concepts to established industry ontologies.

### Context 
You are performing a mapping task using the relation equivalentProperty

### Properties to Map (Extracted from Document)
{"\n".join([f"Predicate name: {predicate_legal["label"]} Description: {predicate_legal["description"]}" for predicate_legal in predicates_to_output.values()])}

### External Reference Ontologies
Below are the definitions from external standards (e.g., AIRO, DPV, or the EU AI Act official terminology):
{"\n".join([f"Predicate URI: {iri}  label: {predicate_external["label"]} Description: {predicate_external["comment"]}" for iri, predicate_external in predicates_external.items()])}


### Task 
1. Analyze each Extracted Properties against the External Reference Ontologies.
2. Identify if an equivalence can be found.
3. If no clear relationship exists, do not guess; omit the relation."""

In [ ]:
predicates_to_output

{'0': {'label': 'receives',
  'description': 'An AI system takes in input data to generate outputs.',
  'domain': 'AI system',
  'range': 'input data',
  'source': 'Generated'},
 '1': {'label': 'uses',
  'description': 'Emotion recognition systems, biometric categorisation systems, and remote biometric identification systems use biometric data for their operations.',
  'domain': 'AI system',
  'range': 'biometric data',
  'source': 'Generated'},
 '2': {'label': 'isA',
  'description': 'A general-purpose AI system is a specific type of AI system.',
  'domain': 'general-purpose AI system',
  'range': 'AI system',
  'source': 'Generated'},
 '3': {'label': 'isA',
  'description': 'An emotion recognition system is a specific type of AI system.',
  'domain': 'emotion recognition system',
  'range': 'AI system',
  'source': 'Generated'},
 '4': {'label': 'isA',
  'description': 'A biometric categorisation system is a specific type of AI system.',
  'domain': 'biometric categorisation system',


In [ ]:
import agent_alignment_predicates

model_anchor_predicates = agent_alignment_predicates.set_structured_output(model=model)
res_anchoring_predicates = agent_alignment_predicates.apply_prompt(model_anchor_predicates, prompt)

In [ ]:
anchoring_predicates = dict()
id = 0
for anchor in res_anchoring_predicates.anchoring_relations:
    anchoring_predicates[id] = {
        "sub":anchor.subject,
        "obj":anchor.object,
        "iri":anchor.object_iri,
        "justification":anchor.justification
    }

    id+=1

with open(f"{PATH_OUTPUT}anchoring_predicates.json", "w", encoding="UTF-8") as file_json:
    json.dump(anchoring_predicates, file_json, indent=4)

In [ ]:
anchoring_predicates

{0: {'sub': 'AI system',
  'obj': 'input data',
  'iri': 'https://w3id.org/airo#hasInput',
  'justification': "The 'receives' predicate, when an AI system is the subject and input data is the object, directly corresponds to the AI system having that input data, making 'airo:hasInput' an equivalent property."},
 1: {'sub': 'AI system',
  'obj': 'risks to health, safety, or fundamental rights',
  'iri': 'https://w3id.org/airo#hasRisk',
  'justification': "The 'poses' predicate indicates that an AI system brings forth or is associated with risks, which is semantically equivalent to the AI system 'having risk'."},
 2: {'sub': 'High-risk AI system',
  'obj': 'instructions for use',
  'iri': 'https://w3id.org/airo#hasDocumentation',
  'justification': "The 'accompaniedBy' predicate, in the context of an AI system and instructions for use, means the system comes with or has this documentation, which is directly equivalent to 'airo:hasDocumentation'."},
 3: {'sub': 'AI system',
  'obj': 'risk 

# Generate the real ontology

In [11]:
base_URI = "<http://example.org/eu-ai-act#"
def set_uri(element:str)->str:
   
    return f":{element.replace(" ", "_").replace("(","").replace(")","").replace(".","").replace(",","")}"

In [12]:
with open(f"{PATH_OUTPUT}classes.json", "r", encoding="UTF-8") as f_r:
    classes = json.load(f_r)

with open(f"{PATH_OUTPUT}predicates.json", "r", encoding="UTF-8") as f_r:
    predicates = json.load(f_r)

with open(f"{PATH_OUTPUT}anchored_classes.json", "r", encoding="UTF-8") as f_r:
    anchored_classes = json.load(f_r)

merged_predicates = dict()
for predicate in predicates.values():
    label = predicate["label"] 
    if label not in merged_predicates:
        merged_predicates[label] = {
            "label": label,
            "description":{predicate["description"]},
            "domain": {predicate["domain"]},
            "range": {predicate["range"]}
        }

    else:
        merged_predicates[label]["description"].add(predicate["description"])
        merged_predicates[label]["domain"].add(predicate["domain"])
        merged_predicates[label]["range"].add(predicate["description"])

In [ ]:
with open(f"{PATH_OUTPUT}ontology_generated.ttl","w", encoding="UTF-8") as file_ontology:
    file_ontology.write("""@prefix : <http://example.org/eu-ai-act#> .
@prefix dc: <http://purl.org/dc/elements/1.1/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix xml: <http://www.w3.org/XML/1998/namespace> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix airo: <https://w3id.org/airo#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix terms: <http://purl.org/dc/terms/> .
@prefix prov: <https://www.w3.org/TR/prov-o/> .
@prefix schema: <http://schema.org/> .
                        
""")

    file_ontology.write("""#################################################################
# Classes description
#################################################################
                    
""")
    for class_information in classes.values():
        uri_class = set_uri(class_information["label"])
        file_ontology.write(f'{uri_class} rdf:type owl:Class;\n')
        file_ontology.write(f'\t rdfs:label "{class_information["label"]}"@en;\n')
        file_ontology.write(f'\t rdfs:comment "{class_information["description"].replace("\n", " ")}"@en;\n')
        file_ontology.write(f'\t prov:wasAttributedTo {"'document'"if class_information["source"] == "Doc" else "'LLM'"}.\n')
        file_ontology.write("\n")


    file_ontology.write("""#################################################################
# Predicate description
#################################################################

""")
    for predicate_information in merged_predicates.values():
        uri_predicate = set_uri(predicate_information["label"])
        file_ontology.write(f'{uri_predicate} rdf:type owl:ObjectProperty;\n')
        file_ontology.write(f'\t rdfs:label "{predicate_information["label"]}"@en;\n')
        file_ontology.write(f'\t rdfs:comment "{list(predicate_information["description"])[0]}"@en;\n')

        if len(domain_values := predicate_information["domain"]) > 0:
            first = True
            for domain_value in domain_values:
                if domain_value:
                    if not first:
                        file_ontology.write(",\n")
                        file_ontology.write("\t\t")
                    else:
                        file_ontology.write(f'\t schema:domainIncludes ')
                        first = False
                    file_ontology.write(f"{set_uri(domain_value)}")
            if not first:
                file_ontology.write(";\n")

        if len(range_values := predicate_information["range"]) > 0:
            first = True
            for range_value in range_values:
                if range_value:
                    if not first:
                        file_ontology.write(",\n")
                        file_ontology.write("\t\t")
                    else:
                        file_ontology.write(f'\t schema:rangeIncludes ')
                        first = False
                    file_ontology.write(f"{set_uri(range_value)}")
            if not first:
                file_ontology.write(";\n")
        file_ontology.write(f'\t prov:wasAttributedTo "LLM".\n')
        file_ontology.write("\n")

    
    file_ontology.write("""#################################################################
# Classes alignments
#################################################################

""")
    for anchored_class in anchored_classes.values():
        uri_class_gen = set_uri(anchored_class["sub"])
        uri_class_external = anchored_class["iri"]

        file_ontology.write(f"{uri_class_gen} ")
        if anchored_class["prop"] == "equivalentClass":
            file_ontology.write(f"owl:equivalentClass ")
        else:
            file_ontology.write(f"rdfs:subClassOf ")
        
        file_ontology.write(f"<{anchored_class["iri"]}>.\n\n")

# Subclass structure

In [ ]:
import agent_generate_hierarchy

prompt = """### Role
You are a Senior Ontologist and Semantic Architect specializing in Description Logic (DL).

### Context
You are provided with a Turtle (.ttl) draft of an ontology for the EU AI Act. This draft includes:
1.  **Core Classes**: Extracted from the legal text with rdfs:comment definitions.
2.  **Alignments**: Anchored mappings to external ontologies (airo, vair).
3.  **Predicates**: Initial object properties, specifically the "isA" property which contains latent hierarchical data.

### Task
Your goal is to define a formal Class Hierarchy by asserting `rdfs:subClassOf` relations between the local classes (prefixed with `:`). 

### Instructions
1.  **Analyze "isA" Mapping**: Review the `schema:rangeIncludes` list for the `<http://example.org/eu-ai-act#isA>` predicate. Convert these logic-based descriptions into formal `rdfs:subClassOf` triples.
2.  **Deduce Internal Subclasses**: Based on both the `rdfs:label` and the `rdfs:comment` definitions, identify "parent-child" relationships. 
    * *Example*: Since an "importer" is defined as a type of "operator," you must assert `:importer rdfs:subClassOf :operator .`
3.  **Refine Categorization**: Group specific systems (e.g., `emotion_recognition_system`) under general categories (e.g., `AI_system`).
4.  **Consolidate Alignments**: Ensure that if `:A rdfs:subClassOf :B` and `:B owl:equivalentClass airo:C`, then the hierarchy remains consistent with the external anchor.

### Output Requirements
Return a structured collection of relations where each entry represents a single `rdfs:subClassOf` link.

### Input Ontology
# """

ontology_str = str()
with open(f"{PATH_OUTPUT}ontology_generated.ttl", "r", encoding="UTF-8") as f_r:
    ontology_str = f_r.read()

In [4]:
import agent_generate_hierarchy

prompt = """### Role
You are a Senior Ontologist and Logical Architect specializing in Description Logic (DL). Your expertise is in converting flat legal terminology into formal taxonomic hierarchies.

### Context
You are provided with a Turtle (.ttl) draft of an ontology for the EU AI Act. This document contains:
1. **Core Classes**: Extracted legal concepts with labels and definitions (rdfs:comment).
2. **Existing Predicates**: Initial relationships found between these concepts.

### Task
Your goal is to build a formal **Subsumption Hierarchy**. You must analyze the definitions of the provided classes to determine which are "Specific Species" and which are "General Genera."

### Extraction Logic
Identify pairs where a **Precise Class** (Specific) belongs to a **General Class** (Parent). Use the following legal logic:
* **Actors**: Map specific roles (e.g., "importer") to their broader categories (e.g., "operator").
* **Systems**: Map specialized AI applications (e.g., "emotion recognition system") to the root "AI_system".
* **Data**: Map specific data types (e.g., "biometric data") to broader classifications (e.g., "personal data").

### Output Requirements
For every relationship found, populate the following fields in the structured output:
* **subClass**: The name of the specific, more precise class.
* **higherClass**: The name of the most immediate general parent class.

### Constraints
1. **Minimal Distance**: Link to the closest possible parent (e.g., "Real-time remote biometric identification" should link to "Remote biometric identification", not directly to "AI system").
2. **Definition-Based**: Use the `rdfs:comment` to justify the hierarchy. If the text says "X means a type of Y," X is the subClass and Y is the higherClass.

### Input Ontology (Turtle)
# """

ontology_str = str()
with open(f"{PATH_OUTPUT}ontology_generated.ttl", "r", encoding="UTF-8") as f_r:
    ontology_str = f_r.read()

In [7]:
print(prompt)
model_generate_hierarchy = agent_generate_hierarchy.set_structured_output(model=model)
res_predicates_generation = agent_generate_hierarchy.apply_prompt(model_generate_hierarchy, prompt, ontology_str)

### Role
You are a Senior Ontologist and Logical Architect specializing in Description Logic (DL). Your expertise is in converting flat legal terminology into formal taxonomic hierarchies.

### Context
You are provided with a Turtle (.ttl) draft of an ontology for the EU AI Act. This document contains:
1. **Core Classes**: Extracted legal concepts with labels and definitions (rdfs:comment).
2. **Existing Predicates**: Initial relationships found between these concepts.

### Task
Your goal is to build a formal **Subsumption Hierarchy**. You must analyze the definitions of the provided classes to determine which are "Specific Species" and which are "General Genera."

### Extraction Logic
Identify pairs where a **Precise Class** (Specific) belongs to a **General Class** (Parent). Use the following legal logic:
* **Actors**: Map specific roles (e.g., "importer") to their broader categories (e.g., "operator").
* **Systems**: Map specialized AI applications (e.g., "emotion recognition system

In [8]:
hierarchy_res = {(res.subClass, res.higherClass, res.justification)for res in res_predicates_generation.relations_ontology}
hierarchy_res

with open(f"{PATH_OUTPUT}hierarchy_2.tuple", "w", encoding="UTF-8") as f_out:
    for (sub, higher, justification) in hierarchy_res:
        f_out.write(f"{sub}\t{higher}\t{justification}\n")

In [14]:
with open(f"{PATH_OUTPUT}ontology_generated.ttl","a", encoding="UTF-8") as file_ontology:
    file_ontology.write("""#######################
# SubClassOf
########################""")
    
    for relation in res_predicates_generation.relations_ontology:
        sub = relation.subClass
        high = relation.higherClass

        file_ontology.write(f"{set_uri(sub)} rdfs:subClassOf {set_uri(high)}. \n\n")